# Transformer Summarization - Analysis

Qualitative and quantitative analysis of summarization outputs, error analysis, and comparison across model variants.

## Part-3:
### Installing libraries and packages

In [1]:
!pip install -q transformers accelerate bitsandbytes torch huggingface_hub
from huggingface_hub import login # authenticate with HuggingFace
import torch
# importing transformer components for loading Llama-3
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import pandas as pd
import random
import warnings
warnings.filterwarnings('ignore')
# checking GPU is available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu") # checking GPU
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("Warning: Enable GPU")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.4/59.4 MB 45.8 MB/s eta 0:00:00
Using device: cuda
GPU: NVIDIA A100-SXM4-80GB


## Authenticate with HuggingFace

In [2]:
HF_TOKEN = os.environ.get("HUGGINGFACE_TOKEN", "")
login(token=HF_TOKEN)
print("\nSetup complete")


Setup complete


## Loading dataset and selecting same essay

In [3]:
df = pd.read_csv('/content/essays.csv') # loading essays dataset
test_df = df.iloc[1800:].copy() # taking only test set, indices 1800 onwards

random.seed(40)  # same seed as Part 2
test_texts = test_df['essay'].tolist()
references = test_df['description'].tolist()
sample_indices = random.sample(range(len(test_texts)), 2) # randomly selecting 2 essays

essay_1 = test_texts[sample_indices[0]] # extracting two selected essays
essay_2 = test_texts[sample_indices[1]]
ground_truth_1 = references[sample_indices[0]] # getting their ground truth summaries
ground_truth_2 = references[sample_indices[1]]

print(f"\nLoaded dataset: {len(df)} essays")
print(f"Selected indices: {sample_indices}")
print(f"\nEssay 1: {len(essay_1)} chars \nGround truth: {ground_truth_1}")
print(f"Essay 2: {len(essay_2)} chars\nGround truth: {ground_truth_2}")


Loaded dataset: 2235 essays
Selected indices: [234, 296]

Essay 1: 23356 chars 
Ground truth: Animals have thoughts, feelings and personality. Why have we taken so long to catch up with animal consciousness?
Essay 2: 16963 chars
Ground truth: UFO sightings are down. Ghosts are in decline. Are we more discerning now, or just afraid to trust anything?


## Loading Meta-LLaMA-3-8B model and tokenizer

In [4]:
MODEL_NAME = "meta-llama/Meta-Llama-3-8B-Instruct" # loading Llama-3-8B with 4-bit quantization

print(f"Loading {MODEL_NAME}.")
quantization_config = BitsAndBytesConfig(load_in_4bit=True,bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",bnb_4bit_use_double_quant=True) # configure 4-bit quantization

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME) # loading tokenizer
if tokenizer.pad_token is None: # setting up padding token
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

model = AutoModelForCausalLM.from_pretrained(MODEL_NAME,quantization_config=quantization_config,
    device_map="auto",trust_remote_code=True,torch_dtype=torch.float16) # loading model

Loading meta-llama/Meta-Llama-3-8B-Instruct.


tokenizer_config.json:   0%|          | 0.00/51.0k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/73.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/654 [00:00<?, ?B/s]

...

model-00004-of-00004.safetensors:   0%|          | 0.00/1.17G [00:00<?, ?B/s]

model-00003-of-00004.safetensors:   0%|          | 0.00/4.92G [00:00<?, ?B/s]

model-00001-of-00004.safetensors:   0%|          | 0.00/4.98G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/187 [00:00<?, ?B/s]

## Defining prompts

In [15]:
PROMPT_1 = """Summarize the following essay in one or two sentences.
Essay:
{essay}
Summary:""" # Direct instruction

PROMPT_2 = """You're writing a quick summary of this essay for someone who hasn't read it yet.
Read the following essay and create a summary that:
1.⁠ Gets the main point or argument across clearly
2.⁠ Is about 1-2 sentences long
3.⁠ Sounds natural and easy to understand
4.⁠ Sticks to the big ideas, not small details
Essay:
{essay}
Summary:""" # Structured requirements

PROMPT_3 = """Here are examples of essays and their summaries:
Example 1:
Essay: Online shopping has taken over everything. You can buy groceries, furniture, even cars without leaving your couch. It started with Amazon but now every store has an app. The upside is you save time and sometimes money with online deals. The downside is you can't actually see or touch what you're buying, return shipping is a hassle, and a lot of people impulse-buy stuff they don't need just because it's so easy. Plus all those delivery trucks are bad for the environment.
Summary: Online shopping offers convenience and potential savings but encourages impulse purchases, creates return complications, and contributes to environmental concerns through excessive packaging and delivery emissions.

Example 2:
Essay: Coffee culture has gotten out of control lately. A regular coffee used to cost like a dollar but now people are paying seven bucks for a fancy latte with oat milk and special foam. Coffee shops are packed with people working on laptops for hours, and there's a new trendy cafe opening every week. Some people say it's about the experience and the Instagram photos, not just the caffeine. But it's making coffee less accessible for people who just want a simple affordable drink without all the fancy extras.
Summary: The rise of specialty coffee culture has transformed coffee from an affordable daily beverage into an expensive lifestyle product, prioritizing aesthetic experience over accessibility and value.

Now, following the same style, summarize this essay:
Essay:
{essay}
Summary:""" # Few shot learning

PROMPT_4 = """You're texting someone who asked what the essay was about, but you're trying to keep it under 160 characters because you don't want to send a long message. Give them the quick version.

Here’s the essay:
{essay}
Your text message summary:""" # In-Context learning

prompts = { # storing all prompts in a dictionary for easy iteration
    "Prompt 1 (Direct)": PROMPT_1,
    "Prompt 2 (Structured)": PROMPT_2,
    "Prompt 3 (Few-Shot)": PROMPT_3,
    "Prompt 4 (In-Context)": PROMPT_4}
print(f"Created {len(prompts)} prompts:")
for i, name in enumerate(prompts.keys(), 1):
    print(f"   {i}. {name}")

Created 4 prompts:
   1. Prompt 1 (Direct)
   2. Prompt 2 (Structured)
   3. Prompt 3 (Few-Shot)
   4. Prompt 4 (In-Context)


## Generation Function

In [16]:
def generate_summary(essay, prompt_template, max_new_tokens=100): # generating summary using model
    truncated_essay = essay[:2000] # truncating to 2000 characters
    if len(essay) > 2000:
        truncated_essay += "..."
    full_prompt = prompt_template.format(essay=truncated_essay) # creating full prompt
    inputs = tokenizer(full_prompt, return_tensors="pt", truncation=True, max_length=2048).to(device)  # tokenize
    with torch.no_grad(): # generate
        outputs = model.generate(**inputs,max_new_tokens=max_new_tokens,do_sample=True,
            temperature=0.6,top_p=0.9,top_k=50,num_return_sequences=1,
            pad_token_id=tokenizer.pad_token_id,eos_token_id=tokenizer.eos_token_id,
            repetition_penalty=1.1)
    generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)  # decoding and cleaning
    summary = generated_text[len(full_prompt):].strip() # extracting only the generated part
    summary = summary.split('\n')[0]# taking only the first line
    # removing common prefixes
    for prefix in ["Summary:", "Answer:", "Here's the summary:", "The summary is:", "In summary,"]:
        if summary.startswith(prefix):
            summary = summary[len(prefix):].strip()
    return summary

## Generating all summaries

In [18]:
print("==============")
print("Generating summaries")
print("==============")

results_essay_1 = {} # dictionaries to store results
results_essay_2 = {}
for prompt_name, prompt_template in prompts.items():# looping through each prompt and generating summaries
    print(f"\n{prompt_name}:")
    try: # essay 1
        summary1 = generate_summary(essay_1, prompt_template)
        results_essay_1[prompt_name] = summary1
        print(f"  Essay 1: {summary1[:110]}...")
    except Exception as e: # if generation fails record the error
        results_essay_1[prompt_name] = "[Generation failed]"
        print(f"  Essay 1: Error - {e}")
    try: # essay 2
        summary2 = generate_summary(essay_2, prompt_template)
        results_essay_2[prompt_name] = summary2
        print(f"  Essay 2: {summary2[:110]}...")
    except Exception as e:
        results_essay_2[prompt_name] = "[Generation failed]"
        print(f"  Essay 2: Error - {e}")
print("Summaries generated")

Generating summaries

Prompt 1 (Direct):
  Essay 1: The author recounts their encounter with a flock of semipalmated sandpipers in Jamaica Bay, describing their b...
  Essay 2: The writer describes witnessing a UFO (unidentified flying object) hovering in the night sky outside their bed...

Prompt 2 (Structured):
  Essay 1: This essay is about the author's encounter with a flock of semipalmated sandpipers, a type of shorebird, in Ja...
  Essay 2: The author describes a vivid and unsettling experience they had while lying in bed, where they saw a large, si...

Prompt 3 (Few-Shot):
  Essay 1: The author encounters a flock of semipalmated sandpipers, a species of small shorebirds, while observing them ...
  Essay 2: The author recounts a peculiar event where they witnessed a large, silver, flat-shaped object with flashing li...

Prompt 4 (In-Context):
  Essay 1: "Essay's about a guy who finds a semipalmated sandpiper on Jamaica Bay & watches them for hours, learning abou...
  Essay 2: "Sa

## Displaying results

In [19]:
# Displaying comprehensive comparison
print("\n==============")
print("ESSAY 1 - Comprehensive comparision")
print("===============")
print(f"\nOriginal essay (first 500 chars):\n{essay_1[:500]}...")
print(f"\nGround truth:\n{ground_truth_1}")
print(f"\nGenerated summaries:")
for name, summary in results_essay_1.items(): # showing summary from each prompt
    print(f"\n{name}:\n  {summary}")
print("\n==============")
print("ESSAY 2 - Comprehensive comparision")
print("===============")
print(f"\nOriginal essay (first 500 chars):\n{essay_2[:500]}...")
print(f"\nGround truth:\n{ground_truth_2}")
print(f"\nGenerated summaries:")
for name, summary in results_essay_2.items():
    print(f"\n{name}:\n  {summary}")


ESSAY 1 - Comprehensive comparision

Original essay (first 500 chars):
I met my first semipalmated sandpiper in a crook of Jamaica Bay, an overlooked shore strewn with broken bottles and religious offerings at the edge of New York City. I didn’t know what it was called, this small, dun-and-white bird running the flats like a wind-up toy, stopping to peck mud and racing to join another bird like itself, and then more. Soon a flock formed, several hundred fast-trotting feeders that at some secret signal took flight, wheeling with the flashing synchronisation that res...

Ground truth:
Animals have thoughts, feelings and personality. Why have we taken so long to catch up with animal consciousness?

Generated summaries:

Prompt 1 (Direct):
  The author recounts their encounter with a flock of semipalmated sandpipers in Jamaica Bay, describing their behavior, physical characteristics, and fascinating migration patterns. The essay explores the author's appreciation for these small, yet rema

## Detailed analysis of 2 selected prompts and statistics

In [20]:
selected_prompts = ["Prompt 3 (Few-Shot)", "Prompt 4 (In-Context)"] # detailed comparison
print("\n===============")
print("Detailed analysis of 2 prompts")
print("================")
for essay_num, (essay, gt, results) in enumerate([ # analyzing both essays with the selected prompts
    (essay_1, ground_truth_1, results_essay_1),
    (essay_2, ground_truth_2, results_essay_2)], start=1):
    print("\n===============")
    print(f"Essay {essay_num}")
    print("===============")
    print(f"\nOriginal:\n{essay[:700]}...")
    print(f"\nGround truth:\n{gt}")
    for prompt_name in selected_prompts: # showing results from 2 selected prompts
        print(f"\n{prompt_name}:\n  {results[prompt_name]}")

# Summary statistics
print("\n===============")
print("SUMMARY STATISTICS")
print("===============")
print(f"\n{'Prompt':<30} {'Essay 1':<10} {'Essay 2':<10}")
print("===============")

for name in prompts.keys():# calculating length for each prompt's output
    # getting length (0 if generation failed)
    len1 = len(results_essay_1[name]) if results_essay_1[name] != "[Generation failed]" else 0
    len2 = len(results_essay_2[name]) if results_essay_2[name] != "[Generation failed]" else 0
    print(f"{name:<30} {len1:<10} {len2:<10}")
print(f"\nGround Truth: {len(ground_truth_1):<10} {len(ground_truth_2):<10}")


Detailed analysis of 2 prompts

Essay 1

Original:
I met my first semipalmated sandpiper in a crook of Jamaica Bay, an overlooked shore strewn with broken bottles and religious offerings at the edge of New York City. I didn’t know what it was called, this small, dun-and-white bird running the flats like a wind-up toy, stopping to peck mud and racing to join another bird like itself, and then more. Soon a flock formed, several hundred fast-trotting feeders that at some secret signal took flight, wheeling with the flashing synchronisation that researchers observing starlings have mathematically likened to avalanche formation and liquids turning to gas. Entranced, I spent the afternoon watching them. The birds were too wary to approach, but if ...

Ground truth:
Animals have thoughts, feelings and personality. Why have we taken so long to catch up with animal consciousness?

Prompt 3 (Few-Shot):
  The author encounters a flock of semipalmated sandpipers, a species of small shorebirds, wh